# SC4001 Topic J: Member B training (CLIP probe + ResNet-50)

**Runtime:** set Runtime > Change runtime type > **T4 GPU** before starting.

Run the cells top to bottom. Every output (checkpoints, predictions, experiment log) is saved to
`MyDrive/sc4001_project/results/`, so nothing is lost if Colab disconnects. If it disconnects, rerun
cells 1–3 and then the training cell: runs that already finished are skipped automatically.

### 1. Mount Drive, get the latest code, install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
if os.path.isdir('/content/repo'):
    !git -C /content/repo pull
else:
    !git clone https://github.com/ster2345/SC4001-Grp-Proj.git /content/repo
%cd /content/repo
!pip install -q -r requirements.txt

### 2. Unpack both data versions from Drive to Colab's local disk

In [ ]:
if not os.path.isdir('/content/data/original'):
    !mkdir -p /content/data
    !tar -xf /content/drive/MyDrive/sc4001_project/raw_downloads/original_subset.tar -C /content/data
if not os.path.isdir('/content/data/cleaned'):
    !tar -xf /content/drive/MyDrive/sc4001_project/raw_downloads/cleaned_subset.tar -C /content/data
!python src/check_splits.py --csv splits/splits.csv --root /content/data   # expect: ALL CHECKS PASSED: 5500 images

### 3. Check the GPU

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch runtime to T4 GPU')

### 4. Quick code check (about 1 minute)
Trains on 64 images for 1 epoch, saved to a throwaway folder (not Drive). It only checks that the code runs:
the numbers mean nothing.

In [ ]:
!python src/train_resnet.py --version original --seed 42 --quick --out /content/quick_test
!python src/train_resnet.py --version cleaned  --seed 42 --quick --out /content/quick_test

### 5. CLIP linear probe (both data versions, about 1 minute)
Logistic regression on the saved CLIP features. It prints validation results for each C and picks the best.

In [ ]:
!python src/train_clip_probe.py --version original
!python src/train_clip_probe.py --version cleaned

### 6. ResNet-50: 2 data versions × 3 seeds = 6 runs
Each run prints one line per epoch. Runs whose predictions already exist on Drive are skipped,
so after a disconnect you can just rerun this cell.

In [ ]:
RES = '/content/drive/MyDrive/sc4001_project/results/predictions'
for version in ['original', 'cleaned']:
    for seed in [42, 43, 44]:
        done = os.path.exists(f'{RES}/resnet50_{version}_seed{seed}.csv')
        if done:
            print(f'skip {version} seed {seed}: already done')
            continue
        print(f'===== ResNet-50 | {version} | seed {seed} =====')
        !python src/train_resnet.py --version {version} --seed {seed}

### 7. Experiment log (validation results only) and prediction-file check

In [ ]:
import pandas as pd, glob, sys
sys.path.insert(0, 'src')
from train_utils import check_prediction_file
print(pd.read_csv('/content/drive/MyDrive/sc4001_project/results/experiment_log.csv').to_string())
for f in sorted(glob.glob(f'{RES}/*.csv')):
    check_prediction_file(f)

### 8. Ablation: more epochs (15 instead of 10), 2 versions × 3 seeds (about 30 min)
Checks whether the ResNet was undertrained (some main runs picked epoch 8 of 10). Everything else is unchanged.
Saved as `resnet50_<version>_seed<s>_ep15.*`, so the main results are **not** overwritten.
Already-finished ablation runs are skipped, so you can rerun this cell after a disconnect.

In [ ]:
for version in ['original', 'cleaned']:
    for seed in [42, 43, 44]:
        if os.path.exists(f'{RES}/resnet50_{version}_seed{seed}_ep15.csv'):
            print(f'skip {version} seed {seed} ep15: already done')
            continue
        print(f'===== ABLATION ep15 | ResNet-50 | {version} | seed {seed} =====')
        !python src/train_resnet.py --version {version} --seed {seed} --max-epochs 15 --tag ep15

In [ ]:
# compare main (max 10 epochs) vs ablation (max 15 epochs) on VALIDATION results only
log = pd.read_csv('/content/drive/MyDrive/sc4001_project/results/experiment_log.csv')
r = log[log.model.isin(['resnet50', 'resnet50_ep15'])]
print(r[['model', 'data_version', 'seed', 'best_epoch', 'val_acc', 'val_auc', 'val_loss']].to_string(index=False))